# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/talhahmad-webdev/Machine-Learning-Engineering-internship--FlyrankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Lane: Content Priority Ranking (Scoring)**

The question this lane answers is *"Which content should an editor refresh first?"*,  that's a
"which ones first?" question, which maps to **ranking/scoring**, not classification or
clustering. The deliverable isn't a single yes/no label per item; it's an ordered priority list
across a client's content, driven by a numeric priority score.

- **Decision improved:** which piece of content an editor spends their next hour refreshing.
- **Who acts on it:** a content editor / SEO strategist working a client's content queue.
- **Cost of a wrong call:** editor time wasted refreshing content that was already fine, while a
  genuinely declining, high-traffic page keeps losing clicks unattended.

In [5]:
!git clone https://github.com/talhahmad-webdev/Machine-Learning-Engineering-internship--FlyrankAI.git

fatal: destination path 'Machine-Learning-Engineering-internship--FlyrankAI' already exists and is not an empty directory.


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/Machine-Learning-Engineering-internship--FlyrankAI/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df[['content_id', 'client_id', 'content_type', 'search_volume',
    'avg_position', 'ctr', 'trend_pct', 'trend_direction']].head(5)

(30000, 44)


,content_id,client_id,content_type,search_volume,avg_position,ctr,trend_pct,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,10.0,10.6,0.76,-41.4,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,90.0,20.3,0.05,-57.7,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.0,36.5,0.09,-60.9,down
3,content_331d6c4de07b,client_19581e27de,keyword article,10.0,6.2,0.49,-13.8,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.0,44.0,0.13,-34.7,down


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no `priority` column in the data; it has to be built as a **proxy target**, combining
observed signals about both *how much a page is losing* and *how much opportunity is on the
table*:

- **Decline signal** : `trend_pct` (more negative = losing more traffic momentum)
- **Opportunity signal** :`search_volume` (more searches = more upside) and `competition`
  (lower competition = easier win)
- **Headroom signal** :`avg_position` (worse position = more room to climb), excluding rows
  where `avg_position == 0` (that means *no ranking data*, not "position zero" , a documented
  data-dictionary gotcha)

`priority_score` is a weighted combination (z-scores) of these **observed** columns, not a
hand-picked label. Note `trend_direction` is deliberately excluded as a feature: it's just a
categorical restatement of `trend_pct` and would double-count the same signal.

In [8]:
# Target/proxy inputs: all observed, none derived from a human-defined rule
proxy_inputs = ['trend_pct', 'search_volume', 'competition', 'avg_position']
df[proxy_inputs].describe()

,trend_pct,search_volume,competition,avg_position
count,26612.000000,27532.000000,27532.000000,30000.00000
mean,-4.785969,158.882391,0.146954,16.34238
std,473.861780,1518.270825,0.285241,15.21679
min,-100.000000,0.000000,0.000000,0.00000
25%,-62.600000,0.000000,0.000000,6.20000
50%,-33.500000,10.000000,0.000000,10.80000
75%,0.000000,20.000000,0.130000,22.30000
max,44900.000000,74000.000000,1.000000,245.00000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: precision@K** (checked against `trend_direction`), with Spearman rank correlation as a
secondary sense-check.

Take the top-K items by `priority_score` and ask: of those, how many were actually declining
(`trend_direction == 'down'`)? That's computable today on the starter data as a baseline sanity
check, before any real model exists. "Good" means precision@K clearly above the dataset's overall
down-rate, though see the honest result below for why this single check has limits for *this*
particular proxy target.

**Honest result:** precision@200 came out at 45%, actually *below* the dataset's 56% baseline
down-rate. That's not a bug , `priority_score` intentionally blends opportunity and headroom on
top of decline, so a stable-but-high-opportunity page can outrank a mildly-declining one.
Matching "down" 1:1 was never the goal; this single metric isn't a complete validator for a
proxy this broad. That gap is itself a useful finding, exactly the kind of thing later weeks
(honest validation, leakage hunting) exist to dig into properly, not something to fix by
tweaking weights until the number looks good.

In [9]:
scored = df[df['avg_position'] > 0].copy()

def z(s):
    return (s - s.mean()) / s.std()

decline_signal = z(-scored['trend_pct'].fillna(0))
opportunity_signal = z(scored['search_volume'].fillna(0)) - z(scored['competition'].fillna(0))
headroom_signal = z(scored['avg_position'])

scored['priority_score'] = (
    0.5 * decline_signal + 0.3 * opportunity_signal + 0.2 * headroom_signal
)

top_k = scored.sort_values('priority_score', ascending=False).head(200)
precision_at_k = (top_k['trend_direction'] == 'down').mean()
baseline_rate = (scored['trend_direction'] == 'down').mean()

print(f"precision@200: {precision_at_k:.2%}")
print(f"baseline down-rate: {baseline_rate:.2%}")

# Honest note: priority_score blends opportunity + headroom on top of decline, so it will
# NOT simply match the 'down' rate 1:1 -- that's expected, not a bug. See markdown above.

precision@200: 45.00%
baseline down-rate: 56.45%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one piece of content**: a single pseudonymized article/page (`content_id`),
belonging to one `client_id`, described by its trailing-90-day performance metrics.

In [10]:
# Unit of analysis: one row = one content item
scored[['content_id', 'client_id', 'content_type', 'trend_pct',
        'search_volume', 'competition', 'avg_position', 'priority_score']]\
    .sort_values('priority_score', ascending=False).head(15)

,content_id,client_id,content_type,trend_pct,search_volume,competition,avg_position,priority_score
12140,content_ef99c4abd9ab,client_3fdba35f04,keyword article,3.6,74000.0,0.08,38.5,15.299918
28282,content_454cc6654c6e,client_3fdba35f04,keyword article,-52.8,60500.0,0.11,44.9,12.678596
6972,content_bf67a444faef,client_3fdba35f04,keyword article,-20.0,60500.0,0.11,45.5,12.650515
17907,content_5ec29ae79c60,client_3fdba35f04,keyword article,73.0,60500.0,0.13,49.8,12.583626
18701,content_deb54e9e19cd,client_3fdba35f04,keyword article,-2.5,60500.0,0.13,41.7,12.559580
8055,content_cd6760921db8,client_3fdba35f04,keyword article,-65.0,49500.0,0.08,47.3,10.527221
16005,content_83e3da1394ac,client_19581e27de,keyword article,218.1,49500.0,0.03,65.5,10.510645
22788,content_ee4630879d03,client_3fdba35f04,keyword article,-29.3,49500.0,0.06,25.2,10.217905
13502,content_f76ccf7a7834,client_19581e27de,keyword article,-37.9,49500.0,0.23,9.5,9.836770
15923,content_84fe9d0a707a,client_3fdba35f04,keyword article,-13.5,40500.0,0.10,43.3,8.572765


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (e.g. "flag anything with `trend_pct < -20`") only looks at one signal at a time and
can't trade signals off against each other. In practice:

- A page with a small decline but huge `search_volume` and low `competition` may be a better use
  of an editor's hour than a page with a big decline but tiny traffic potential.
- A page stuck at position 15 with decent volume has more realistic headroom than one stuck at
  position 80.
- These signals interact, and their *relative* importance shifts across 32 different clients , a
  client with mostly new content behaves differently from one with a mature catalog.

Hand-writing an if/else rule that captures these trade-offs for every client would either be too
coarse to be useful or balloon into an unmaintainable pile of thresholds. A learned scoring model
can weigh these signals from data, adapt as patterns shift, and be evaluated honestly (via
precision@K) instead of just trusted on faith.

In [11]:
# Quick illustration: correlation between the three signals shows they don't move together
scored[['trend_pct', 'search_volume', 'competition', 'avg_position']].corr()

,trend_pct,search_volume,competition,avg_position
trend_pct,1.000000,0.002511,0.014446,0.047198
search_volume,0.002511,1.000000,0.050455,0.045336
competition,0.014446,0.050455,1.000000,0.053238
avg_position,0.047198,0.045336,0.053238,1.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.